# Code Testing Notebook

This notebook tests the various classes and functionalities of the psBQP (pseudo-spin Bogoliubov Quasiparticle) Keldysh implementation.

## Table of Contents
1. [NambuKeldyshTensor Class Tests](#nambu-keldysh-tensor)
2. [StateObject Class Tests](#state-object)
3. [EquilibriumSolver Tests](#equilibrium-solver)
4. [UsadelKeldyshEvolution Tests](#usadel-keldysh-evolution)
5. [Thermal Distribution Tests](#thermal-distribution)
6. [Evolution Equations Tests](#evolution-equations)

In [1]:
# Import necessary libraries
import numpy as np
import matplotlib.pyplot as plt
import sys

# Import our classes
from nambu_keldysh_class import NambuKeldyshTensor
from state_object_class import StateObject
from equilibrium_class import EquilibriumSolver
from usadel_keldysh_evolution import UsadelKeldyshEvolution

print("All imports successful!")

All imports successful!


---
## 1. NambuKeldyshTensor Class Tests <a name="nambu-keldysh-tensor"></a>

Test basic operations of the NambuKeldyshTensor class.

### 1.1 Creation and Initialization

In [2]:
# Test creating NambuKeldyshTensor with different Pauli indices
print("Testing NambuKeldyshTensor creation:")

# Create simple test data
test_data = np.array([[1.0 + 0.5j]], dtype=complex)

# Create tensors with different Pauli indices
for pauli_idx in range(4):
    tensor = NambuKeldyshTensor(test_data, pauli_index=pauli_idx)
    print(f"Pauli index {pauli_idx}: shape = {tensor.data.shape}")
    print(f"  Data:\n{tensor.data}")
    print()

Testing NambuKeldyshTensor creation:


TypeError: NambuKeldyshTensor.__init__() got an unexpected keyword argument 'pauli_index'

### 1.2 Pauli Matrix Operations

In [ ]:
# Test that Pauli matrices satisfy expected properties
print("Testing Pauli matrix properties:")

# Create identity and Pauli matrices
data = np.array([[1.0]], dtype=complex)
tau0 = NambuKeldyshTensor(data, pauli_index=0)
tau1 = NambuKeldyshTensor(data, pauli_index=1)
tau2 = NambuKeldyshTensor(data, pauli_index=2)
tau3 = NambuKeldyshTensor(data, pauli_index=3)

# Test τ_i² = I for i = 1,2,3
print("Testing τ_i² = I:")
for i, tau in enumerate([tau1, tau2, tau3], start=1):
    tau_squared = tau @ tau
    print(f"τ_{i}²:")
    print(tau_squared.data[:, :, 0, 0])
    print()

# Test anticommutation relations
print("Testing {τ_1, τ_2} = 0:")
anticomm = tau1 @ tau2 + tau2 @ tau1
print(f"Max value: {np.max(np.abs(anticomm.data))}")
print()

### 1.3 Addition and Scalar Multiplication

In [ ]:
# Test addition and scalar multiplication
print("Testing addition and scalar multiplication:")

# Create test tensors
a = NambuKeldyshTensor(np.array([[2.0]], dtype=complex), pauli_index=1)
b = NambuKeldyshTensor(np.array([[3.0]], dtype=complex), pauli_index=1)

# Test addition
c = a + b
print(f"a + b:")
print(c.data[:, :, 0, 0])
print()

# Test scalar multiplication
d = a * 2.0
print(f"a * 2.0:")
print(d.data[:, :, 0, 0])
print()

# Test subtraction
e = b - a
print(f"b - a:")
print(e.data[:, :, 0, 0])
print()

### 1.4 Matrix Multiplication (@)

In [ ]:
# Test matrix multiplication in Nambu space
print("Testing matrix multiplication:")

# Create gap tensor: Δ = Δ_r τ_1 + Δ_i τ_2
Delta_r = 0.5
Delta_i = 0.3
gap_tensor = (NambuKeldyshTensor(np.array([[Delta_r]], dtype=complex), pauli_index=1) +
              NambuKeldyshTensor(np.array([[Delta_i]], dtype=complex), pauli_index=2))

print(f"Gap tensor (Δ = {Delta_r} + {Delta_i}i):")
print(gap_tensor.data[:, :, 0, 0])
print()

# Multiply with τ_3
tau3 = NambuKeldyshTensor(np.array([[1.0]], dtype=complex), pauli_index=3)
result = tau3 @ gap_tensor
print(f"τ_3 @ Δ:")
print(result.data[:, :, 0, 0])
print()

### 1.5 Trace Operation

In [ ]:
# Test trace operation
print("Testing trace operation:")

# Create a simple 2x2 matrix in time
N_t = 3
test_data = np.random.randn(2, 2, N_t, N_t) + 1j * np.random.randn(2, 2, N_t, N_t)
tensor = NambuKeldyshTensor(test_data)

# Trace with different Pauli indices
for pauli in [0, 1, 2, 3, '-']:
    traced = tensor.trace(pauli_index=pauli)
    print(f"Trace with pauli_index='{pauli}': shape = {traced.shape}")
    if pauli == 0:
        print(f"  Sample value: {traced[0, 0]}")
    print()

### 1.6 Involution Operation

In [ ]:
# Test involution: g^A = -τ_3 (g^R)^† τ_3
print("Testing involution operation:")

# Create a test retarded Green's function
N_t = 2
gr_data = np.random.randn(2, 2, N_t, N_t) + 1j * np.random.randn(2, 2, N_t, N_t)
gr = NambuKeldyshTensor(gr_data)

# Compute involution
ga = gr.involution()

print(f"Original g^R at (0,0):")
print(gr.data[:, :, 0, 0])
print()

print(f"Involution g^A at (0,0):")
print(ga.data[:, :, 0, 0])
print()

# Verify involution property: involution of involution should give back -original
gr_back = ga.involution()
diff = gr_back.data + gr.data
print(f"Max difference (should be ~0): {np.max(np.abs(diff))}")

---
## 2. StateObject Class Tests <a name="state-object"></a>

Test the StateObject container for Green's functions.

### 2.1 Creation and Basic Properties

In [ ]:
# Create a simple StateObject
print("Testing StateObject creation:")

N_t = 5
gr_data = np.random.randn(2, 2, N_t, N_t) + 1j * np.random.randn(2, 2, N_t, N_t)
gk_data = np.random.randn(2, 2, N_t, N_t) + 1j * np.random.randn(2, 2, N_t, N_t)

gr = NambuKeldyshTensor(gr_data)
gk = NambuKeldyshTensor(gk_data)

bcs_coupling = 0.3
grid_params = {'time_duration': 10.0, 'time_sampling': N_t}

state = StateObject(gr, gk, bcs_coupling, grid_params)

print(f"StateObject created with:")
print(f"  g^R shape: {state.gr.data.shape}")
print(f"  g^K shape: {state.gk.data.shape}")
print(f"  BCS coupling: {state.bcs_coupling_constant}")
print(f"  T_max: {state.T_max}")
print(f"  dt: {state.dt}")
print()

### 2.2 Gap History Extraction

In [ ]:
# Test gap history extraction
print("Testing gap history extraction:")

gap_history = state.get_gap_history()

print(f"Gap history shape: {gap_history.shape}")
print(f"Gap values: {gap_history}")
print()

# Plot gap history
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(np.real(gap_history), 'o-', label='Re(Δ)')
plt.plot(np.imag(gap_history), 's-', label='Im(Δ)')
plt.xlabel('Time index')
plt.ylabel('Gap')
plt.legend()
plt.title('Gap History')

plt.subplot(1, 2, 2)
plt.plot(np.abs(gap_history), 'o-')
plt.xlabel('Time index')
plt.ylabel('|Δ|')
plt.title('Gap Magnitude')
plt.tight_layout()
plt.show()

### 2.3 Advanced Green's Function

In [ ]:
# Test computing advanced Green's function
print("Testing advanced Green's function:")

ga = state._r2a()

print(f"g^A shape: {ga.data.shape}")
print(f"g^A at (0,0):")
print(ga.data[:, :, 0, 0])
print()

### 2.4 String Representation

In [ ]:
# Test string representation
print("Testing StateObject string representation:")
print(state)

---
## 3. EquilibriumSolver Tests <a name="equilibrium-solver"></a>

Test the equilibrium solver wrapper.

### 3.1 Initialization

In [ ]:
# Setup grid and system parameters
print("Testing EquilibriumSolver initialization:")

grid_parameters = {
    'omega_sampling': 101,
    'energy_cutoff': 10.0,
    'eta': 0.05
}

system_parameters = {
    'critical_temperature': 1.0,
    'temperature': 0.3
}

optimization_parameters = {
    'max_iterations': 100,
    'tolerance': 1e-6
}

eq_solver = EquilibriumSolver(
    grid_parameters,
    system_parameters,
    optimization_parameters
)

print(f"EquilibriumSolver created successfully")
print(f"  Omega grid size: {len(eq_solver.usadel_solver.w_arr)}")
print(f"  Omega range: [{eq_solver.usadel_solver.w_arr[0]:.2f}, {eq_solver.usadel_solver.w_arr[-1]:.2f}]")
print()

### 3.2 Compute Equilibrium (Small Test)

In [ ]:
# Compute equilibrium Green's function (this may take some time)
print("Computing equilibrium Green's function...")
print("(This may take a minute)")

try:
    gr_eq = eq_solver.compute_equilibrium_gr(
        temperature=0.3,
        Q=0.0,
        compute_gk=False
    )
    
    print(f"Equilibrium g^R computed successfully")
    print(f"  Type: {type(gr_eq)}")
    print()
except Exception as e:
    print(f"Error computing equilibrium: {e}")
    import traceback
    traceback.print_exc()

---
## 4. UsadelKeldyshEvolution Tests <a name="usadel-keldysh-evolution"></a>

Test the main evolution class.

### 4.1 Initialization

In [ ]:
# Setup parameters for evolution
print("Testing UsadelKeldyshEvolution initialization:")

grid_parameters = {
    'time_sampling': 50,
    'time_duration': 20.0
}

system_parameters = {
    'critical_temperature': 1.0,
    'temperature': 0.3,
    'eta': 0.1
}

evolution = UsadelKeldyshEvolution(
    grid_parameters,
    system_parameters
)

print(f"UsadelKeldyshEvolution created successfully")
print(f"  Time grid: {evolution.ntpoints} points")
print(f"  Time range: [{evolution.time_grid[0]:.2f}, {evolution.time_grid[-1]:.2f}]")
print(f"  dt: {evolution.delta_t:.4f}")
print(f"  Omega grid: {len(evolution.omega_grid)} points")
print(f"  Energy cutoff: {evolution.energy_cutoff:.2f}")
print(f"  eta: {evolution.eta:.4f}")
print()

### 4.2 BCS Helper Methods

In [ ]:
# Test BCS helper methods
print("Testing BCS helper methods:")

gap_constant = evolution.get_bcs_gap_constant()
bcs_ratio = evolution.get_bcs_ratio()
bcs_coupling = evolution._get_BCS_coupling()

print(f"BCS gap constant: {gap_constant:.4f}")
print(f"BCS ratio Δ(0)/T_c: {bcs_ratio:.4f}")
print(f"BCS coupling λ: {bcs_coupling:.4f}")
print()

# Expected values
print(f"Expected BCS gap constant: ~1.134")
print(f"Expected BCS ratio: ~1.764")
print()

---
## 5. Thermal Distribution Tests <a name="thermal-distribution"></a>

Test thermal occupation function generation.

### 5.1 Generate Thermal Distribution

In [ ]:
# Generate thermal distribution
print("Testing thermal distribution generation:")

temperature = 0.3
evolution.get_thermal_occupation(temperature)

print(f"Thermal distribution generated")
print(f"  Shape: {evolution.thermal_dist.shape}")
print(f"  Temperature: {temperature}")
print()

# Check that F(τ) is antisymmetric: F(-τ) = -F(τ)
N_t = evolution.ntpoints
F = evolution.thermal_dist

print("Checking antisymmetry F(-τ) = -F(τ):")
# For diagonal, τ = 0, should be purely imaginary
print(f"  F(0,0) [should be ~pure imaginary]: {F[N_t//2, N_t//2]}")
print()

# Check a few off-diagonal elements
for offset in [1, 5, 10]:
    if N_t//2 + offset < N_t and N_t//2 - offset >= 0:
        F_plus = F[N_t//2, N_t//2 + offset]
        F_minus = F[N_t//2, N_t//2 - offset]
        print(f"  F(τ={offset*evolution.delta_t:.2f}): {F_plus:.4f}")
        print(f"  F(τ={-offset*evolution.delta_t:.2f}): {F_minus:.4f}")
        print(f"  Sum (should be ~0): {F_plus + F_minus}")
        print()

### 5.2 Visualize Thermal Distribution

In [ ]:
# Visualize F(t, t')
print("Visualizing thermal distribution F(t, t'):")

F = evolution.thermal_dist
time_grid = evolution.time_grid

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Real part
im1 = axes[0].imshow(np.real(F), extent=[time_grid[0], time_grid[-1], time_grid[-1], time_grid[0]], 
                     cmap='RdBu', aspect='auto')
axes[0].set_xlabel('t\'')
axes[0].set_ylabel('t')
axes[0].set_title('Re[F(t, t\')]')
plt.colorbar(im1, ax=axes[0])

# Imaginary part
im2 = axes[1].imshow(np.imag(F), extent=[time_grid[0], time_grid[-1], time_grid[-1], time_grid[0]], 
                     cmap='RdBu', aspect='auto')
axes[1].set_xlabel('t\'')
axes[1].set_ylabel('t')
axes[1].set_title('Im[F(t, t\')]')
plt.colorbar(im2, ax=axes[1])

# F as function of τ = t - t' (along diagonal direction)
# Extract diagonal slice
mid_idx = len(time_grid) // 2
F_slice = F[mid_idx, :]
tau_values = time_grid[mid_idx] - time_grid

axes[2].plot(tau_values, np.real(F_slice), 'o-', label='Re[F(τ)]', markersize=3)
axes[2].plot(tau_values, np.imag(F_slice), 's-', label='Im[F(τ)]', markersize=3)
axes[2].set_xlabel('τ = t - t\'')
axes[2].set_ylabel('F(τ)')
axes[2].set_title(f'F(τ) at t={time_grid[mid_idx]:.2f}')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 6. Evolution Equations Tests <a name="evolution-equations"></a>

Test the discretized evolution equations (without running full evolution).

### 6.1 Test Evolution with Mock State

Create a mock state and test one timestep evolution.

In [ ]:
# Create a mock state for testing evolution
print("Testing evolution equations with mock state:")

# Use smaller grid for testing
N_t_test = 10

# Create mock Green's functions (random for testing)
gr_mock = NambuKeldyshTensor(np.random.randn(2, 2, N_t_test, N_t_test) + 
                             1j * np.random.randn(2, 2, N_t_test, N_t_test))
gk_mock = NambuKeldyshTensor(np.random.randn(2, 2, N_t_test, N_t_test) + 
                             1j * np.random.randn(2, 2, N_t_test, N_t_test))

bcs_coupling = evolution._get_BCS_coupling()
grid_params_test = {'time_duration': evolution.tmax, 'time_sampling': N_t_test}

mock_state = StateObject(gr_mock, gk_mock, bcs_coupling, grid_params_test)

print(f"Mock state created with N_t = {N_t_test}")
print(f"Gap history: {mock_state.get_gap_history()}")
print()

In [ ]:
# Test g^R evolution for one timestep
print("Testing g^R evolution:")

try:
    gr_new, gr_diag_new = evolution._evolve_gr_by_one_timestep(mock_state)
    print(f"g^R evolution successful!")
    print(f"  Type of gr_new: {type(gr_new)}")
    print(f"  Type of gr_diag_new: {type(gr_diag_new)}")
    print()
except Exception as e:
    print(f"Error in g^R evolution: {e}")
    import traceback
    traceback.print_exc()
    print()

In [ ]:
# Test g^K evolution for one timestep
print("Testing g^K evolution:")

try:
    gk_new, gk_diag_new = evolution._evolve_gk_by_one_timestep(mock_state)
    print(f"g^K evolution successful!")
    print(f"  Type of gk_new: {type(gk_new)}")
    print(f"  Type of gk_diag_new: {type(gk_diag_new)}")
    print()
except Exception as e:
    print(f"Error in g^K evolution: {e}")
    import traceback
    traceback.print_exc()
    print()

---
## Summary

This notebook tests the core functionalities of the psBQP Keldysh implementation. Run all cells to verify that:

1. ✓ NambuKeldyshTensor operations work correctly
2. ✓ StateObject properly manages Green's functions
3. ✓ EquilibriumSolver wraps old code correctly
4. ✓ UsadelKeldyshEvolution initializes properly
5. ✓ Thermal distribution is generated correctly
6. ✓ Evolution equations can be evaluated

Add more tests as needed for specific functionalities!